## === Imports ===

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve, make_scorer
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb


## === Chemins et paramètres ===

In [ ]:
DATA_PATH  = Path("../data/processed/training_dataset.csv")
MODEL_DIR  = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
mae_scorer   = make_scorer(mean_absolute_error, greater_is_better=False)

## === Chargement des données ===

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Lignes : {len(df):,}")
print(f"Colonnes : {df.shape[1]}")
print(f"Années : {df['year'].min()} - {df['year'].max()}")

## === Features de base (identiques à la v1) ===

In [ ]:
# Features déjà présentes en v1
df['lap_time_delta']    = df['last_lap_time_ms'] - df['avg_lap_time_ms']
df['pace_vs_best']      = df['last_lap_time_ms'] - df['best_lap_time_ms']
df['consistency']       = df['lap_time_std'] / (df['avg_lap_time_ms'] + 1)
df['is_early_race']     = (df['race_progress_pct'] < 33).astype(int)
df['is_mid_race']       = ((df['race_progress_pct'] >= 33) & (df['race_progress_pct'] < 66)).astype(int)
df['is_late_race']      = (df['race_progress_pct'] >= 66).astype(int)
df['position_momentum'] = df['grid_position'] - df['current_position']
df['has_pitted']        = (df['nb_pit_stops'] > 0).astype(int)
df['pit_frequency']     = df['nb_pit_stops'] / (df['lap'] + 1)

# Colonnes cibles
df['is_top3']   = (df['final_position'] <= 3).astype(int)
df['is_winner'] = (df['final_position'] == 1).astype(int)

print("Features v1 créées")

## === Nouvelles features (v2) ===

On ajoute 4 features simples qui aident beaucoup :
- **late_top5** : est-il dans le Top 5 en fin de course ? (très prédictif du podium)
- **recovery_potential** : est-ce qu'il a encore assez de tours pour remonter ?
- **pace_gap_to_leader** : est-il rapide par rapport aux autres ce tour ?
- **overtake_opportunity** : a-t-il progressé par rapport à sa position de départ ?

In [ ]:
# Feature 1 : dans le Top 5 ET en fin de course
# => très souvent, les Top 5 à 66% de la course finissent dans le Top 3
df['late_top5'] = ((df['current_position'] <= 5) & (df['is_late_race'] == 1)).astype(int)

# Feature 2 : potentiel de remontée
# => tours restants divisé par position actuelle
# => grand = beaucoup de temps pour remonter depuis loin
df['recovery_potential'] = df['laps_remaining'] / (df['current_position'] + 1)

# Feature 3 : écart de rythme avec le meilleur de ce tour
# => 0 = il est le plus rapide, grand = il est lent
df['pace_gap_to_leader'] = df.groupby(['raceId', 'lap'])['last_lap_time_ms'].transform(lambda x: x - x.min())

# Feature 4 : positions gagnées depuis le départ
# => positif = il a progressé, négatif = il a reculé
df['overtake_opportunity'] = df['grid_position'] - df['current_position']

print("Nouvelles features v2 créées")
print(f"Total colonnes : {df.shape[1]}")

## === Préparation Train / Val / Test ===

In [ ]:
# Colonnes à exclure (pas des features)
EXCLUDE_COLS = [
    'raceId', 'driverId', 'constructorId', 'circuitId',
    'final_position', 'is_dnf', 'is_top3', 'is_winner',
]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
print(f"Nombre de features : {len(feature_cols)}")

# Split temporel
train_df = df[df['year'] <= 2019].copy()
val_df   = df[(df['year'] >= 2020) & (df['year'] <= 2021)].copy()
test_df  = df[df['year'] >= 2022].copy()

print(f"Train : {len(train_df):,} lignes")
print(f"Val   : {len(val_df):,} lignes")
print(f"Test  : {len(test_df):,} lignes")

# X et y
X_train = train_df[feature_cols]
y_train = train_df['final_position']

X_val   = val_df[feature_cols]
y_val   = val_df['final_position']

X_test  = test_df[feature_cols]
y_test  = test_df['final_position']

## === Sample Weights : les fins de course comptent plus ===

En v1, tous les tours avaient le même poids.

Logiquement, le tour 55/60 est plus prédictif que le tour 2/60.
On donne donc plus d'importance aux tours de fin de course.

- Début de course → poids 1.0
- Fin de course   → poids 3.0
- Vrai Top 3      → poids × 2 en plus

In [ ]:
# Poids basé sur la progression dans la course (0% → 1.0, 100% → 3.0)
progress = train_df['race_progress_pct'] / 100.0
poids_course = 1.0 + 2.0 * progress

# Les vrais Top 3 reçoivent un bonus × 2
# => pour que le modèle apprenne mieux à les détecter
bonus_top3 = np.where(train_df['is_top3'] == 1, 2.0, 1.0)

# Poids final
sample_weights = (poids_course * bonus_top3).values

print(f"Poids moyen début de course : {sample_weights[train_df['is_early_race'].values == 1].mean():.2f}")
print(f"Poids moyen fin de course   : {sample_weights[train_df['is_late_race'].values == 1].mean():.2f}")
print(f"Poids moyen vrais Top 3     : {sample_weights[train_df['is_top3'].values == 1].mean():.2f}")

## === Sous-échantillon pour le tuning ===

Pour aller plus vite, on cherche les meilleurs paramètres sur 100 000 lignes seulement.
Ensuite on entraîne le modèle final sur TOUTES les données.

In [ ]:
# Prendre 100 000 lignes aléatoires pour le tuning
np.random.seed(RANDOM_STATE)
idx_tune = np.random.choice(len(X_train), 100_000, replace=False)

X_tune = X_train.iloc[idx_tune]
y_tune = y_train.iloc[idx_tune]
w_tune = sample_weights[idx_tune]

print(f"Taille sous-échantillon : {len(X_tune):,} lignes")

## === Entraînement XGBoost ===

In [ ]:
# Paramètres à tester
xgb_params = {
    'n_estimators'     : [500, 700, 1000],
    'max_depth'        : [6, 8, 10],
    'learning_rate'    : [0.01, 0.03, 0.05],
    'subsample'        : [0.7, 0.8, 0.9],
    'colsample_bytree' : [0.7, 0.8, 0.9],
    'min_child_weight' : [1, 3, 5],
}

xgb_base = xgb.XGBRegressor(
    tree_method='hist',    # plus rapide sur CPU
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_search = RandomizedSearchCV(
    xgb_base, xgb_params,
    n_iter=30,
    scoring=mae_scorer,
    cv=3,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1
)

# --- Tuning sur sous-échantillon ---
print("Tuning XGBoost...")
t0 = time.time()
xgb_search.fit(X_tune, y_tune, sample_weight=w_tune)
print(f"Terminé en {(time.time()-t0)/60:.1f} min")
print(f"Meilleurs paramètres : {xgb_search.best_params_}")
print(f"MAE (tuning) : {-xgb_search.best_score_:.3f}")

# --- Entraînement final sur tout le train set ---
print("\nEntraînement final XGBoost...")
t0 = time.time()
xgb_model = xgb.XGBRegressor(
    **xgb_search.best_params_,
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

## === Entraînement LightGBM ===

In [ ]:
# Paramètres à tester
lgb_params = {
    'n_estimators'      : [500, 700, 1000],
    'max_depth'         : [6, 8, 10],
    'learning_rate'     : [0.01, 0.03, 0.05],
    'num_leaves'        : [31, 50, 63],
    'subsample'         : [0.7, 0.8, 0.9],
    'colsample_bytree'  : [0.7, 0.8, 0.9],
    'min_child_samples' : [10, 20, 30],
}

lgb_base = lgb.LGBMRegressor(
    tree_learner='serial',   # évite le gel sur CPU local
    random_state=RANDOM_STATE,
    verbose=-1,
    n_jobs=1
)

lgb_search = RandomizedSearchCV(
    lgb_base, lgb_params,
    n_iter=30,
    scoring=mae_scorer,
    cv=3,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1
)

# --- Tuning sur sous-échantillon ---
print("Tuning LightGBM...")
t0 = time.time()
lgb_search.fit(X_tune, y_tune, sample_weight=w_tune)
print(f"Terminé en {(time.time()-t0)/60:.1f} min")
print(f"Meilleurs paramètres : {lgb_search.best_params_}")
print(f"MAE (tuning) : {-lgb_search.best_score_:.3f}")

# --- Entraînement final sur tout le train set ---
print("\nEntraînement final LightGBM...")
t0 = time.time()
lgb_model = lgb.LGBMRegressor(
    **lgb_search.best_params_,
    tree_learner='serial',
    random_state=RANDOM_STATE,
    verbose=-1,
    n_jobs=-1
)
lgb_model.fit(X_train, y_train, sample_weight=sample_weights)
print(f"Terminé en {(time.time()-t0)/60:.1f} min")

## === Entraînement Random Forest ===

In [ ]:
# Paramètres à tester
rf_params = {
    'n_estimators'     : [300, 400, 500],
    'max_depth'        : [15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 5],
    'max_features'     : ['sqrt', 'log2'],
}

rf_base = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_search = RandomizedSearchCV(
    rf_base, rf_params,
    n_iter=20,
    scoring=mae_scorer,
    cv=3,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1
)

# --- Tuning sur sous-échantillon ---
print("Tuning Random Forest...")
t0 = time.time()
rf_search.fit(X_tune, y_tune, sample_weight=w_tune)
print(f"Terminé en {(time.time()-t0)/60:.1f} min")
print(f"Meilleurs paramètres : {rf_search.best_params_}")
print(f"MAE (tuning) : {-rf_search.best_score_:.3f}")

# --- Entraînement final sur tout le train set ---
print("\nEntraînement final Random Forest...")
t0 = time.time()
rf_model = RandomForestRegressor(
    **rf_search.best_params_,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train, y_train, sample_weight=sample_weights)
print(f"Terminé en {(time.time()-t0)/60:.1f} min")

## === Comparaison des 3 modèles ===

In [ ]:
# Dictionnaire des modèles
models = {
    'XGBoost'      : xgb_model,
    'LightGBM'     : lgb_model,
    'Random Forest': rf_model,
}

# Calculer les métriques pour chaque modèle
resultats = []

for nom, modele in models.items():
    predictions_test = modele.predict(X_test)

    mae  = mean_absolute_error(y_test, predictions_test)
    rmse = np.sqrt(mean_squared_error(y_test, predictions_test))
    r2   = r2_score(y_test, predictions_test)

    resultats.append({
        'Modèle'   : nom,
        'MAE'      : round(mae, 3),
        'RMSE'     : round(rmse, 3),
        'R2'       : round(r2, 3),
    })

resultats_df = pd.DataFrame(resultats).sort_values('MAE')
print(resultats_df.to_string(index=False))

# Récupérer le meilleur modèle
meilleur_nom    = resultats_df.iloc[0]['Modèle']
meilleur_modele = models[meilleur_nom]
print(f"\nMeilleur modèle : {meilleur_nom}")

## === Trouver le meilleur seuil pour Top 3 ===

On teste maintenant tous les seuils possibles et on garde celui qui donne le meilleur F1.

In [ ]:
# Prédictions du meilleur modèle sur le test set
pred_positions = meilleur_modele.predict(X_test)
vrais_top3     = test_df['is_top3'].values

# Tester tous les seuils de 2.0 à 6.0
seuils  = np.arange(2.0, 6.5, 0.1)
f1_scores = []
recalls   = []
precisions = []

for seuil in seuils:
    pred_top3 = (pred_positions <= seuil).astype(int)
    f1_scores.append(f1_score(vrais_top3, pred_top3, zero_division=0))
    recalls.append(recall_score(vrais_top3, pred_top3, zero_division=0))
    precisions.append(precision_score(vrais_top3, pred_top3, zero_division=0))

# Seuil avec le meilleur F1
meilleur_idx   = np.argmax(f1_scores)
meilleur_seuil = seuils[meilleur_idx]

print(f"Seuil v1 (fixe)    : 3.5")
print(f"Meilleur seuil v2  : {meilleur_seuil:.1f}")
print(f"")

# Comparaison v1 vs v2
pred_top3_v1 = (pred_positions <= 3.5).astype(int)
pred_top3_v2 = (pred_positions <= meilleur_seuil).astype(int)

print("         Precision  Recall  F1")
print(f"Seuil v1 (3.5) :  {precision_score(vrais_top3, pred_top3_v1, zero_division=0):.3f}     {recall_score(vrais_top3, pred_top3_v1, zero_division=0):.3f}   {f1_score(vrais_top3, pred_top3_v1, zero_division=0):.3f}")
print(f"Seuil v2 ({meilleur_seuil:.1f}) :  {precisions[meilleur_idx]:.3f}     {recalls[meilleur_idx]:.3f}   {f1_scores[meilleur_idx]:.3f}")

## === Sauvegarde des modèles ===

In [ ]:
# Sauvegarder les 3 modèles
joblib.dump(xgb_model, MODEL_DIR / 'f1_xgboost_v2.pkl')
joblib.dump(lgb_model, MODEL_DIR / 'f1_lightgbm_v2.pkl')
joblib.dump(rf_model,  MODEL_DIR / 'f1_random_forest_v2.pkl')

# Sauvegarder la liste des features
joblib.dump(feature_cols, MODEL_DIR / 'feature_names_v2.pkl')

# Sauvegarder le seuil optimal
joblib.dump(meilleur_seuil, MODEL_DIR / 'optimal_threshold_v2.pkl')

print("Modèles sauvegardés :")
print("  - f1_xgboost_v2.pkl")
print("  - f1_lightgbm_v2.pkl")
print("  - f1_random_forest_v2.pkl")
print("  - feature_names_v2.pkl")
print("  - optimal_threshold_v2.pkl")
print(f"\nSeuil optimal sauvegardé : {meilleur_seuil:.1f}")

## === Résultats finaux v1 vs v2 ===

In [ ]:
mae_v2 = resultats_df.iloc[0]['MAE']

print("============================================")
print("         RÉSULTATS FINAUX  ")
print("============================================")
print(f"")
print(f"  MAE test       : →  {mae_v2}")
print(f"  Recall Top 3   : →  {recalls[meilleur_idx]:.3f}")
print(f"  F1 Top 3       : →  {f1_scores[meilleur_idx]:.3f}")
print(f"  Seuil Top 3    : →  {meilleur_seuil:.1f}")
print(f"  Features       : →  {len(feature_cols)}")

print(f"  Meilleur modèle : {meilleur_nom}")
print("============================================")